# Step 12e — Recalculate Cohen's Kappa with Synthetic Instances

> Recalculate Cohen's Kappa by merging the original manual evaluation and the newly added synthetic evaluations evaluated by Balancer A (A2) and Balancer B (A3).

## Inputs
| Path | Format | Source |
|---|---|---|
| `data/11_manual_evaluation.csv` | CSV | Original manual evaluation |
| `data/12e_sinth_manual_evaluation.csv` | CSV | Synthetic evaluations (Balancer A & B) |

## Outputs
| Path | Format | Description |
|---|---|---|
| `data/12e_manual_evaluation_combined.csv` | CSV | Combined evaluation (Original + Synthetic) |
| `data/12e_manual_evaluation_combined_with_one_occurrence_by_smell.csv` | CSV | Filtered combined evaluations (single-occurrence only) |
| `data/12e_irr_results.csv` | CSV | Recalculated Cohen's Kappa results per-pair and overall |
| `data/step12e_report.json` | JSON | Recalculation details, hashes and filter statistics |



In [1]:
"""Step 12e - Recalculate evaluation summary (original + synthetic instances)

Recalculates Cohen's Kappa by merging the original manual evaluation and the
newly added synthetic evaluations evaluated by Balancer A (A2) and Balancer B
(A3).

The single-occurrence filter and the IRR math are shared with Step 11
(``11_summarize_evaluation``) via :mod:`dataset_utils.irr` - one implementation,
so the two steps can no longer drift. This step adds only
:func:`combine_evaluations` (concatenate original + synthetic rows) on top.
"""

import argparse
import csv
import json
import sys
from pathlib import Path

# Guarded __file__ hack: make dataset_utils importable when run as a plain script
# without an installed wheel. Harmless in Jupyter (__file__ undefined). Do NOT remove.
try:
    sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
except NameError:
    pass

from dataset_utils.hashing import compute_file_hash
from dataset_utils.irr import compute_irr, filter_one_occurrence, print_summary, write_irr_csv
from dataset_utils.notebook_bootstrap import setup

setup()

# Input Paths
ORIGINAL_EVAL_PATH = Path("data/11_manual_evaluation.csv")
SYNTH_EVAL_PATH = Path("data/12e_sinth_manual_evaluation.csv")

# Output Paths
COMBINED_EVAL_PATH = Path("data/12e_manual_evaluation_combined.csv")
FILTERED_PATH = Path("data/12e_manual_evaluation_combined_with_one_occurrence_by_smell.csv")
OUTPUT_CSV_PATH = Path("data/12e_irr_results.csv")
REPORT_PATH = Path("data/step12e_report.json")


def combine_evaluations(orig_path: Path, synth_path: Path, combined_path: Path) -> None:
    """Concatenate original and synthetic manual evaluations into a single CSV."""
    with orig_path.open(encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)

    with synth_path.open(encoding="utf-8", newline="") as f:
        synth_reader = csv.DictReader(f)
        synth_rows = list(synth_reader)

    # Sanity check: headers must match
    if reader.fieldnames != synth_reader.fieldnames:
        raise ValueError(
            f"CSV header mismatch!\nOriginal: {reader.fieldnames}\nSynthetic: {synth_reader.fieldnames}"
        )

    combined_rows = rows + synth_rows

    combined_path.parent.mkdir(parents=True, exist_ok=True)
    with combined_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(combined_rows)

    print(f"Combined evaluation written to {combined_path}")
    print(f"  Original rows : {len(rows)}")
    print(f"  Synthetic rows: {len(synth_rows)}")
    print(f"  Combined rows : {len(combined_rows)}")


def run(
    orig_path: Path = ORIGINAL_EVAL_PATH,
    synth_path: Path = SYNTH_EVAL_PATH,
    combined_path: Path = COMBINED_EVAL_PATH,
    filtered_path: Path = FILTERED_PATH,
    output_csv: Path = OUTPUT_CSV_PATH,
    report_path: Path = REPORT_PATH,
) -> dict:
    # 1. Combine evaluations (original + synthetic).
    combine_evaluations(orig_path, synth_path, combined_path)

    # 2. Filter the combined evaluation -> single-occurrence CSV (shared w/ Step 11).
    filter_stats = filter_one_occurrence(combined_path, filtered_path)

    # 3. IRR over the filtered rows (shared w/ Step 11).
    result = compute_irr(filtered_path)
    write_irr_csv(result, output_csv)

    report = {
        **result,
        "filter": filter_stats,
        "orig_input_sha256": compute_file_hash(orig_path),
        "synth_input_sha256": compute_file_hash(synth_path),
        "combined_file": str(combined_path),
        "combined_sha256": compute_file_hash(combined_path),
        "filtered_file": str(filtered_path),
        "filtered_sha256": compute_file_hash(filtered_path),
        "output_file": str(output_csv),
        "output_sha256": compute_file_hash(output_csv),
    }
    report_path.parent.mkdir(parents=True, exist_ok=True)
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    return report


def _parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Recalculate manual evaluation with synthetic instances."
    )
    p.add_argument("--orig", type=Path, default=ORIGINAL_EVAL_PATH)
    p.add_argument("--synth", type=Path, default=SYNTH_EVAL_PATH)
    p.add_argument("--combined", type=Path, default=COMBINED_EVAL_PATH)
    p.add_argument("--filtered", type=Path, default=FILTERED_PATH)
    p.add_argument("--output-csv", type=Path, default=OUTPUT_CSV_PATH)
    p.add_argument("--report", type=Path, default=REPORT_PATH)
    return p.parse_args()


def main() -> None:
    args = _parse_args()
    report = run(
        args.orig,
        args.synth,
        args.combined,
        args.filtered,
        args.output_csv,
        args.report,
    )
    fs = report["filter"]
    print("Recalculate evaluation complete.")
    print(f"  Original Input : {args.orig}")
    print(f"  Synthetic Input: {args.synth}")
    print(f"  Combined CSV   : {args.combined}")
    print(f"  Filtered CSV   : {args.filtered}")
    print(f"  IRR CSV        : {args.output_csv}")
    print(f"  Report         : {args.report}")
    print(
        f"  Filter         : {fs['n_in']} -> {fs['n_out']} rows "
        f"({fs['n_dropped']} multi-occurrence present dropped)"
    )
    print_summary(report)


if __name__ == "__main__":
    main()


Combined evaluation written to data/12e_manual_evaluation_combined.csv
  Original rows : 1846
  Synthetic rows: 49
  Combined rows : 1895
Recalculate evaluation complete.
  Original Input : data/11_manual_evaluation.csv
  Synthetic Input: data/12e_sinth_manual_evaluation.csv
  Combined CSV   : data/12e_manual_evaluation_combined.csv
  Filtered CSV   : data/12e_manual_evaluation_combined_with_one_occurrence_by_smell.csv
  IRR CSV        : data/12e_irr_results.csv
  Report         : data/step12e_report.json
  Filter         : 1895 -> 1818 rows (77 multi-occurrence present dropped)

  Pair             N   Kappa Interpretation      Concordance
  ------------ ----- ------- ------------------ ------------
  A1-A4          710  0.7670 Substantial              88.45%
  A2-A3           79  0.8910 Almost perfect           94.94%
  A5-A8          438  0.7853 Substantial              89.27%
  A6-A7          591  0.9492 Almost perfect           97.46%
  ------------ ----- ------- ------------------